# 31 — Multi-NR LGBM: ChemBERTa-MTR + ESM-2 Protein Embeddings

Same multi-NR structure as nb30 but uses **ChemBERTa-MTR CLS embeddings** (768-dim) as
compound features instead of Morgan+RDKit fingerprints.

- Compound features: ChemBERTa-MTR CLS (768-dim)
  - PXR CRC train/test: loaded from pre-computed cache
  - ChEMBL NR compounds: extracted on-the-fly
- Protein features: ESM-2 (320-dim)
- Total input: **1088 features**

In [1]:
import sys, warnings, json
sys.path.insert(0, '../src')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import lightgbm as lgb
import torch
from transformers import AutoTokenizer, AutoModel

from pxr.data import load_train, load_test
from pxr.chem import bemis_murcko
from pxr.eval import scaffold_kfold_indices, compute_metrics, rae as rae_fn
from pxr.paths import DATA_PROCESSED, SUBMISSIONS

SEED = 42
N_FOLDS = 5
BATCH_SIZE = 64
# Must match the model used for chemberta_mtr_train/test_emb.npy (cached by nb14)
CHEMBERTA_MODEL = 'deepchem/ChemBERTa-77M-MTR'

LGBM_PARAMS = dict(
    n_estimators=1200, num_leaves=64, learning_rate=0.04,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=0.2,
    min_child_samples=10, n_jobs=4, verbose=-1
)

TARGET_WEIGHTS = {
    'PXR':   1.0,
    'VDR':   0.50,
    'FXR':   0.30,
    'LXRa':  0.25,
    'RXRa':  0.25,
    'PPARg': 0.15,
    'PPARa': 0.15,
}

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'device: {DEVICE}')
print('Setup complete.')

device: cpu
Setup complete.


## 1. Load data and protein embeddings

In [2]:
train = load_train()
te    = load_test()
nr    = pd.read_parquet('../data/external/chembl_nr_targets.parquet')

# Load ESM-2 protein embeddings
esm2_arr = np.load(DATA_PROCESSED / 'nr_esm2_embeddings.npy')   # (n_proteins, 320)
with open(DATA_PROCESSED / 'nr_esm2_names.json') as f:
    esm2_names = json.load(f)
esm2_lookup = {name: esm2_arr[i] for i, name in enumerate(esm2_names)}
PROTEIN_DIM = esm2_arr.shape[1]  # 320

print(f'CRC train: {len(train):,}  |  ChEMBL NR: {len(nr):,}  |  Test: {len(te):,}')
print(f'ESM-2 protein dim: {PROTEIN_DIM}')

CRC train: 4,139  |  ChEMBL NR: 11,511  |  Test: 513
ESM-2 protein dim: 320


## 2. Load cached ChemBERTa-MTR embeddings for CRC train/test

In [3]:
cache_tr = DATA_PROCESSED / 'chemberta_mtr_train_emb.npy'
cache_te = DATA_PROCESSED / 'chemberta_mtr_test_emb.npy'

assert cache_tr.exists(), f'Missing cache: {cache_tr} — run nb13 first'
assert cache_te.exists(), f'Missing cache: {cache_te} — run nb13 first'

X_tr_chem = np.load(cache_tr).astype(np.float32)  # (4139, 768)
X_te_chem = np.load(cache_te).astype(np.float32)  # (513, 768)

CHEM_DIM = X_tr_chem.shape[1]  # 768
TOTAL_DIM = CHEM_DIM + PROTEIN_DIM  # 1088

print(f'Train ChemBERTa shape: {X_tr_chem.shape}')
print(f'Test  ChemBERTa shape: {X_te_chem.shape}')
print(f'Total feature dim: {TOTAL_DIM} = {CHEM_DIM} (chem) + {PROTEIN_DIM} (protein)')

Train ChemBERTa shape: (4139, 384)
Test  ChemBERTa shape: (513, 384)
Total feature dim: 704 = 384 (chem) + 320 (protein)


## 3. Extract ChemBERTa-MTR embeddings for ChEMBL NR compounds

In [4]:
nr_filt = nr[(nr['pec50'] >= 3.0) & (nr['pec50'] <= 10.0)].copy()
nr_filt = nr_filt[nr_filt['target_name'].isin(esm2_lookup)].copy().reset_index(drop=True)
nr_filt['weight'] = nr_filt['target_name'].map(TARGET_WEIGHTS).fillna(0.1)
print(f'ChEMBL NR after quality filter: {len(nr_filt):,}')

NR_CACHE = DATA_PROCESSED / 'chemberta_mtr_nr_emb.npy'

if NR_CACHE.exists():
    X_nr_chem_all = np.load(NR_CACHE).astype(np.float32)
    print(f'Loaded ChemBERTa NR cache: {X_nr_chem_all.shape}')
    valid_nr_mask = np.ones(len(nr_filt), dtype=bool)
    X_nr_chem = X_nr_chem_all
else:
    print(f'Extracting ChemBERTa embeddings for {len(nr_filt):,} ChEMBL NR compounds...')
    tokenizer = AutoTokenizer.from_pretrained(CHEMBERTA_MODEL)
    model = AutoModel.from_pretrained(CHEMBERTA_MODEL).to(DEVICE)
    model.eval()

    def extract_chemberta_batch(smiles_list, tokenizer, model, device, max_length=128):
        """Extract ChemBERTa CLS embeddings for a batch of SMILES."""
        embeddings = []
        for i in range(0, len(smiles_list), BATCH_SIZE):
            batch = smiles_list[i:i+BATCH_SIZE]
            try:
                inputs = tokenizer(
                    batch,
                    return_tensors='pt',
                    padding=True,
                    truncation=True,
                    max_length=max_length,
                )
                inputs = {k: v.to(device) for k, v in inputs.items()}
                with torch.no_grad():
                    out = model(**inputs)
                # CLS token is at position 0
                cls_emb = out.last_hidden_state[:, 0, :].cpu().numpy()  # (batch, 768)
                embeddings.append(cls_emb)
            except Exception as e:
                print(f'  Batch {i//BATCH_SIZE} error: {e} — filling with zeros')
                embeddings.append(np.zeros((len(batch), 768), dtype=np.float32))
            if (i // BATCH_SIZE) % 20 == 0:
                print(f'  Processed {i+len(batch):,} / {len(smiles_list):,}')
        return np.vstack(embeddings).astype(np.float32)

    nr_smiles = nr_filt['smiles'].tolist()
    X_nr_chem_all = extract_chemberta_batch(nr_smiles, tokenizer, model, DEVICE)
    np.save(NR_CACHE, X_nr_chem_all)
    print(f'Cached to {NR_CACHE}  shape={X_nr_chem_all.shape}')

    # Identify zero rows (failed SMILES) — treat as invalid
    valid_nr_mask = ~np.all(X_nr_chem_all == 0, axis=1)
    X_nr_chem = X_nr_chem_all

nr_filt_valid = nr_filt[valid_nr_mask].reset_index(drop=True)
X_nr_chem = X_nr_chem_all[valid_nr_mask]
print(f'Valid NR compounds: {len(nr_filt_valid):,}')

ChEMBL NR after quality filter: 11,496
Loaded ChemBERTa NR cache: (11496, 384)
Valid NR compounds: 11,496


## 4. Build multi-NR feature matrices

In [5]:
y_tr = train['pec50'].values
pxr_emb = esm2_lookup['PXR']  # (320,)

# CRC train: [ChemBERTa | PXR protein embedding]
pxr_tile_tr = np.tile(pxr_emb, (len(X_tr_chem), 1))   # (4139, 320)
X_tr_full = np.hstack([X_tr_chem, pxr_tile_tr])         # (4139, 1088)
w_tr = np.ones(len(y_tr))

# ChEMBL NR: [ChemBERTa | per-target ESM-2]
nr_protein_embs = np.stack(
    [esm2_lookup[t] for t in nr_filt_valid['target_name']], axis=0
)  # (n_nr, 320)
X_nr_full = np.hstack([X_nr_chem, nr_protein_embs])      # (n_nr, 1088)
y_nr = nr_filt_valid['pec50'].values
w_nr = nr_filt_valid['weight'].values

# Test: [ChemBERTa | PXR protein embedding]
pxr_tile_te = np.tile(pxr_emb, (len(X_te_chem), 1))    # (513, 320)
X_te_full = np.hstack([X_te_chem, pxr_tile_te])          # (513, 1088)

print(f'Train (CRC): {X_tr_full.shape}')
print(f'NR augment:  {X_nr_full.shape}')
print(f'Test:        {X_te_full.shape}')

Train (CRC): (4139, 704)
NR augment:  (11496, 704)
Test:        (513, 704)


## 5. Scaffold 5-fold CV on CRC train

In [6]:
scaffolds = train['smiles'].map(bemis_murcko).tolist()
splits = scaffold_kfold_indices(scaffolds, n_splits=N_FOLDS, seed=SEED)

oof = np.full(len(y_tr), np.nan)
fold_metrics = []

for fold_i, (tr_idx, va_idx) in enumerate(splits):
    X_fold = np.vstack([X_tr_full[tr_idx], X_nr_full])
    y_fold = np.concatenate([y_tr[tr_idx], y_nr])
    w_fold = np.concatenate([w_tr[tr_idx], w_nr])

    m = lgb.LGBMRegressor(**LGBM_PARAMS)
    m.fit(X_fold, y_fold, sample_weight=w_fold)

    oof[va_idx] = m.predict(X_tr_full[va_idx])
    met = compute_metrics(y_tr[va_idx], oof[va_idx])
    fold_metrics.append(met)
    print(f'  Fold {fold_i+1}: RAE={met["RAE"]:.4f}  Spearman={met["Spearman"]:.4f}')

oof_rae = rae_fn(y_tr, oof)
cv_df = pd.DataFrame(fold_metrics)
print(f'\nOOF RAE (global): {oof_rae:.4f}')
print(f'Mean fold RAE:    {cv_df["RAE"].mean():.4f} +/- {cv_df["RAE"].std():.4f}')
print(f'Mean Spearman:    {cv_df["Spearman"].mean():.4f}')

np.save(DATA_PROCESSED / 'oof_chemberta_esm2_nr.npy', oof)

  Fold 1: RAE=0.5458  Spearman=0.7282


  Fold 2: RAE=0.6294  Spearman=0.6511


  Fold 3: RAE=0.6616  Spearman=0.6321


  Fold 4: RAE=0.6094  Spearman=0.6694


  Fold 5: RAE=0.6739  Spearman=0.6267

OOF RAE (global): 0.6186
Mean fold RAE:    0.6240 +/- 0.0507
Mean Spearman:    0.6615


## 6. Full retrain + test predictions + submission

In [7]:
X_full = np.vstack([X_tr_full, X_nr_full])
y_full = np.concatenate([y_tr, y_nr])
w_full = np.concatenate([w_tr, w_nr])

final_m = lgb.LGBMRegressor(**LGBM_PARAMS)
final_m.fit(X_full, y_full, sample_weight=w_full)

te_preds = np.clip(final_m.predict(X_te_full), y_tr.min() - 0.5, y_tr.max() + 0.5)
np.save(DATA_PROCESSED / 'te_chemberta_esm2_nr.npy', te_preds)

sub = pd.DataFrame({
    'Molecule Name': te['name'].values,
    'SMILES':        te['smiles'].values,
    'pEC50':         te_preds,
})
assert len(sub) == 513 and sub['pEC50'].notna().all(), 'Submission validation failed'

out_path = SUBMISSIONS / '31_chemberta_esm2_multinr.csv'
sub.to_csv(out_path, index=False)
print(f'Saved: {out_path}')
print(f'OOF RAE: {oof_rae:.4f}')
print(f'Test pEC50: min={te_preds.min():.2f}  median={np.median(te_preds):.2f}  max={te_preds.max():.2f}')
print(sub['pEC50'].describe().round(3))

Saved: D:\Users\ashenoy00000\.windsurf\OpenADMET-pxr-challenge\submissions\31_chemberta_esm2_multinr.csv
OOF RAE: 0.6186
Test pEC50: min=2.84  median=4.78  max=5.71
count    513.000
mean       4.680
std        0.536
min        2.842
25%        4.338
50%        4.783
75%        5.081
max        5.715
Name: pEC50, dtype: float64
